# 03 — Нейросетевые признаки

In [1]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import random
import subprocess
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 260824
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("Нужна CUDA-сборка PyTorch и NVIDIA GPU")

DEVICE = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_MB = int(
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()[0]
)

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print("GPU memory MB:", GPU_MEMORY_MB)


torch: 2.12.1+cu126
CUDA: 12.6
GPU: NVIDIA GeForce RTX 4070 SUPER
GPU memory MB: 12282


In [2]:
EXPECTED_USERS = 250_000
HORIZON = 30
LOOKBACK = 365
RECENT_DAYS = 42
PATCH_DAYS = 7
LONG_DAYS = 364
N_PATCHES = LONG_DAYS // PATCH_DAYS
BRIDGE_DIM = 24

V9_CANONICAL_POOL = [
    date(2025, 2, 13),
    date(2025, 3, 26),
    date(2025, 5, 7),
    date(2025, 6, 18),
    date(2025, 7, 30),
    date(2025, 9, 10),
    date(2025, 10, 22),
    date(2025, 11, 12),
    date(2025, 12, 3),
    date(2025, 12, 15),
    date(2026, 1, 14),
]

BUDGET = {
    "pretrain_steps": 400,
    "dense_steps": 300,
    "batch": 256,
    "infer_chunk": 2048,
}

PRETRAIN_LR = 3e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
SCALER_SAMPLE_PAIRS = 1_000_000

W_RECON = 1.00
W_ANY30 = 0.20
W_FREQ = 0.12
W_ACTIVE = 0.05
W_GAP = 0.10
W_REPEAT_GAP = 0.08
W_FIRST_MARK = 0.10
W_LEVEL3 = 0.06
W_DENSE_Z = 0.35

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_ROOT = PROJECT_DIR / "artifacts" / "neural_features"
CACHE_DIR = ARTIFACT_ROOT / "_cache" / "daily_cube"
BRIDGE_DIR = ARTIFACT_ROOT / "representation"

for p in [ARTIFACT_ROOT, CACHE_DIR, BRIDGE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("\n".join(str(p) for p in paths))

TRAIN_PATH = first_existing(
    DATA_DIR / "train.parquet",
    PROJECT_DIR / "train.parquet",
)
SAMPLE_PATH = first_existing(
    DATA_DIR / "sample_submit.csv",
    DATA_DIR / "sample_submission.csv",
    PROJECT_DIR / "sample_submit.csv",
    PROJECT_DIR / "sample_submission.csv",
)

data = pl.read_parquet(TRAIN_PATH)
if data.schema["event_date"] != pl.Date:
    data = data.with_columns(pl.col("event_date").cast(pl.Date))

submit_template = pl.read_csv(SAMPLE_PATH).select("user_id")
all_users = submit_template.unique(maintain_order=True)

assert all_users.height == EXPECTED_USERS
assert all_users["user_id"].n_unique() == EXPECTED_USERS

min_date = data["event_date"].min()
max_date = data["event_date"].max()

assert max_date == date(2026, 2, 13)

print("train:", data.shape)
print("period:", min_date, "->", max_date)


train: (30631006, 18)
period: 2025-01-01 -> 2026-02-13


## Данные

In [3]:
BINARY_COLS = [
    'search', 'cat',
    'has_search_to_cart', 'has_search_to_ord',
    'has_cat_to_cart', 'has_cat_to_ord',
]
FLOW_COLS = [
    'search_to_cart', 'search_to_ord', 'cat_to_cart', 'cat_to_ord',
    'to_cart', 'to_ord', 'searches',
]
MONETARY_COLS = ['gmv_search', 'gmv_cat', 'gmv']
RAW_COLS = BINARY_COLS + FLOW_COLS + MONETARY_COLS
assert len(RAW_COLS) == 16
missing = [c for c in RAW_COLS if c not in data.columns]
if missing:
    raise RuntimeError(f'Missing raw daily columns required by the reviewed 16-channel contract: {missing}')

N_DAYS = (max_date - min_date).days + 1
DATE_TO_INDEX = {min_date + timedelta(days=i): i for i in range(N_DAYS)}
USER_MAP = all_users.with_row_index('__uidx')

X_PATH = CACHE_DIR/'daily_x_log_fp16.dat'
GMV_PATH = CACHE_DIR/'daily_gmv_raw_fp32.dat'
ORD_PATH = CACHE_DIR/'daily_to_ord_raw_fp16.dat'
ROW_PATH = CACHE_DIR/'daily_row_present_u8.dat'
META_PATH = CACHE_DIR/'cube_meta.json'

cube_contract = {
    'users': EXPECTED_USERS, 'days': N_DAYS, 'min_date': str(min_date), 'max_date': str(max_date),
    'raw_cols': RAW_COLS, 'x_dtype': 'float16', 'gmv_dtype': 'float32', 'ord_dtype': 'float16',
}
contract_hash = hashlib.sha256(json.dumps(cube_contract, sort_keys=True).encode()).hexdigest()[:16]

need_build = True
if META_PATH.exists() and all(p.exists() for p in [X_PATH, GMV_PATH, ORD_PATH, ROW_PATH]):
    old = json.loads(META_PATH.read_text())
    need_build = old.get('contract_hash') != contract_hash

if need_build:
    print('Building daily cube...')
    for p in [X_PATH, GMV_PATH, ORD_PATH, ROW_PATH]:
        if p.exists(): p.unlink()
    X_mm = np.memmap(X_PATH, mode='w+', dtype=np.float16, shape=(EXPECTED_USERS, N_DAYS, len(RAW_COLS)))
    GMV_mm = np.memmap(GMV_PATH, mode='w+', dtype=np.float32, shape=(EXPECTED_USERS, N_DAYS))
    ORD_mm = np.memmap(ORD_PATH, mode='w+', dtype=np.float16, shape=(EXPECTED_USERS, N_DAYS))
    ROW_mm = np.memmap(ROW_PATH, mode='w+', dtype=np.uint8, shape=(EXPECTED_USERS, N_DAYS))
    X_mm[:] = 0; GMV_mm[:] = 0; ORD_mm[:] = 0; ROW_mm[:] = 0

    n_unique_ud = data.select(pl.struct(['user_id','event_date']).n_unique().alias('n')).item()
    if int(n_unique_ud) != data.height:
        print('Aggregating duplicate user-date rows...')
        data_daily = (
            data.group_by(['user_id','event_date'])
            .agg([
                *[pl.col(c).max().alias(c) for c in BINARY_COLS],
                *[pl.col(c).sum().alias(c) for c in FLOW_COLS + MONETARY_COLS],
            ])
        )
    else:
        data_daily = data.select(['user_id','event_date',*RAW_COLS])

    joined = (
        data_daily.join(USER_MAP, on='user_id', how='inner', validate='m:1')
        .with_columns((pl.col('event_date') - pl.lit(min_date)).dt.total_days().cast(pl.Int32).alias('__dayidx'))
    )
    del data_daily

    try:
        del data
    except NameError:
        pass
    gc.collect()

    chunk_days = 31
    for s in range(0, N_DAYS, chunk_days):
        e = min(N_DAYS, s + chunk_days)
        part = joined.filter(pl.col('__dayidx').is_between(s, e-1, closed='both'))
        if not part.height:
            continue
        u = part['__uidx'].to_numpy().astype(np.int64, copy=False)
        d = part['__dayidx'].to_numpy().astype(np.int64, copy=False)
        for j, c in enumerate(RAW_COLS):
            v = part[c].fill_null(0).to_numpy().astype(np.float32, copy=False)
            if c not in BINARY_COLS:
                v = np.log1p(np.clip(v, 0, None))
            else:
                v = np.clip(v, 0, 1)
            X_mm[u, d, j] = v.astype(np.float16)
        GMV_mm[u, d] = part['gmv'].fill_null(0).to_numpy().astype(np.float32, copy=False)
        ORD_mm[u, d] = part['to_ord'].fill_null(0).to_numpy().astype(np.float16, copy=False)
        ROW_mm[u, d] = 1
        if s % 93 == 0:
            print(f'  days {s:03d}:{e:03d} | rows={part.height:,}')
    X_mm.flush(); GMV_mm.flush(); ORD_mm.flush(); ROW_mm.flush()
    META_PATH.write_text(json.dumps({**cube_contract, 'contract_hash': contract_hash}, indent=2))
    del joined, X_mm, GMV_mm, ORD_mm, ROW_mm
    gc.collect()
else:
    print('Reusing exact local cube cache:', contract_hash)

X_MM = np.memmap(X_PATH, mode='r', dtype=np.float16, shape=(EXPECTED_USERS, N_DAYS, len(RAW_COLS)))
GMV_MM = np.memmap(GMV_PATH, mode='r', dtype=np.float32, shape=(EXPECTED_USERS, N_DAYS))
ORD_MM = np.memmap(ORD_PATH, mode='r', dtype=np.float16, shape=(EXPECTED_USERS, N_DAYS))
ROW_MM = np.memmap(ROW_PATH, mode='r', dtype=np.uint8, shape=(EXPECTED_USERS, N_DAYS))
try:
    del data
except NameError:
    pass
gc.collect()
print('Cube ready:', X_MM.shape, '| approx GB:', X_MM.nbytes/1e9)


Building daily cube...
  days 000:031 | rows=1,770,329
  days 093:124 | rows=2,007,187
  days 186:217 | rows=2,157,939
  days 279:310 | rows=2,657,849
  days 372:403 | rows=2,863,332
Cube ready: (250000, 409, 16) | approx GB: 3.272


In [4]:
SCALER_PATH = ARTIFACT_ROOT/'frozen_input_scaler.npz'
EARLIEST_SCALE_CUTOFF = date(2025, 6, 18)

def date_idx(d: date) -> int:
    return (d - min_date).days

if SCALER_PATH.exists():
    z = np.load(SCALER_PATH)
    CH_MEAN = z['mean'].astype(np.float32)
    CH_STD = z['std'].astype(np.float32)
else:
    rng = np.random.default_rng(SEED + 17)
    max_di = date_idx(EARLIEST_SCALE_CUTOFF)
    u = rng.integers(0, EXPECTED_USERS, size=SCALER_SAMPLE_PAIRS, dtype=np.int64)
    d = rng.integers(0, max_di + 1, size=SCALER_SAMPLE_PAIRS, dtype=np.int64)
    sample = X_MM[u, d].astype(np.float32)
    CH_MEAN = sample.mean(axis=0).astype(np.float32)
    CH_STD = sample.std(axis=0).astype(np.float32)
    CH_STD = np.where(CH_STD < 1e-4, 1.0, CH_STD).astype(np.float32)
    np.savez_compressed(SCALER_PATH, mean=CH_MEAN, std=CH_STD, cutoff=str(EARLIEST_SCALE_CUTOFF))
    del sample, u, d
    gc.collect()

print('Scaler cutoff:', EARLIEST_SCALE_CUTOFF)
print('Scaler ready')

RAW_C = len(RAW_COLS)

EXTRA_C = 8
INPUT_C = RAW_C + EXTRA_C
IDX_ROW_PRESENT = RAW_C + 5
IDX_VALID = RAW_C + 6
IDX_MASKED = RAW_C + 7

cal = np.zeros((N_DAYS, 4), dtype=np.float32)
for i in range(N_DAYS):
    d = min_date + timedelta(days=i)
    dow_ang = 2*math.pi*d.weekday()/7.0
    doy_ang = 2*math.pi*d.timetuple().tm_yday/365.25
    cal[i] = [math.sin(dow_ang), math.cos(dow_ang), math.sin(doy_ang), math.cos(doy_ang)]
REL_POS = (np.arange(-LOOKBACK+1, 1, dtype=np.float32) / float(LOOKBACK)).reshape(1, LOOKBACK, 1)

def gap_class(gap: np.ndarray) -> np.ndarray:
    g = np.asarray(gap)
    out = np.full(g.shape, 4, dtype=np.int64)
    out[g <= 3] = 0
    out[(g >= 4) & (g <= 7)] = 1
    out[(g >= 8) & (g <= 14)] = 2
    out[(g >= 15) & (g <= 30)] = 3
    return out

class HistoryAccessor:
    def __init__(self):
        self.x = X_MM; self.gmv = GMV_MM; self.ord = ORD_MM; self.row = ROW_MM

    def history_np(self, users: np.ndarray, anchor: date) -> np.ndarray:
        users = np.asarray(users, dtype=np.int64)
        ai = date_idx(anchor)
        idx = np.arange(ai - LOOKBACK + 1, ai + 1, dtype=np.int64)
        valid = (idx >= 0) & (idx < N_DAYS)
        out = np.zeros((len(users), LOOKBACK, INPUT_C), dtype=np.float32)
        if valid.any():
            jj = idx[valid]
            raw = self.x[users[:,None], jj[None,:], :].astype(np.float32)
            raw = np.clip((raw - CH_MEAN[None,None,:]) / CH_STD[None,None,:], -8.0, 8.0)
            out[:, valid, :RAW_C] = raw
            out[:, valid, RAW_C:RAW_C+4] = cal[jj][None,:,:]
            out[:, valid, RAW_C+4] = REL_POS[0, valid, 0][None,:]
            out[:, valid, IDX_ROW_PRESENT] = self.row[users[:,None], jj[None,:]].astype(np.float32)
            out[:, valid, IDX_VALID] = 1.0
        return out

    def labels_np(self, users: np.ndarray, anchor: date) -> dict[str, np.ndarray]:
        users = np.asarray(users, dtype=np.int64)
        ai = date_idx(anchor)
        jj = np.arange(ai+1, ai+HORIZON+1, dtype=np.int64)
        if jj[-1] >= N_DAYS:
            raise ValueError(f'Future horizon not observed for {anchor}')
        g = self.gmv[users[:,None], jj[None,:]].astype(np.float32)
        o = self.ord[users[:,None], jj[None,:]].astype(np.float32)
        row = self.row[users[:,None], jj[None,:]].astype(np.float32)
        order_day = o > 0
        gmv_day = g > 0
        z = np.log1p(np.clip(g.sum(axis=1), 0, None)).astype(np.float32)
        od7 = order_day[:,:7].sum(axis=1).astype(np.float32)
        od14 = order_day[:,:14].sum(axis=1).astype(np.float32)
        od30 = order_day.sum(axis=1).astype(np.float32)
        ad7 = row[:,:7].sum(axis=1).astype(np.float32)
        ad30 = row.sum(axis=1).astype(np.float32)
        any30 = (od30 > 0).astype(np.float32)

        first_gap = np.full(len(users), 31, dtype=np.int16)
        repeat_gap = np.full(len(users), 31, dtype=np.int16)
        has_order = order_day.any(axis=1)
        first_idx = np.argmax(order_day, axis=1)
        first_gap[has_order] = (first_idx[has_order] + 1).astype(np.int16)
        cum_order = np.cumsum(order_day, axis=1)
        second_mask = order_day & (cum_order == 2)
        has_second = second_mask.any(axis=1)
        second_idx = np.argmax(second_mask, axis=1)
        repeat_gap[has_second] = (second_idx[has_second] - first_idx[has_second]).astype(np.int16)

        first_mark = np.zeros(len(users), dtype=np.float32)
        first_mark_valid = gmv_day.any(axis=1).astype(np.float32)
        first_g_idx = np.argmax(gmv_day, axis=1)
        has_g = first_mark_valid > 0.5
        if has_g.any():
            rr = np.flatnonzero(has_g)
            first_mark[rr] = np.log1p(np.clip(g[rr, first_g_idx[rr]], 0, None)).astype(np.float32)

        g_logs = np.log1p(np.clip(g, 0, None)).astype(np.float32)
        cum_g = np.cumsum(gmv_day, axis=1)
        first3_mask = gmv_day & (cum_g <= 3)
        g_count = gmv_day.sum(axis=1)
        level3_valid = (g_count >= 3).astype(np.float32)
        level3 = np.zeros(len(users), dtype=np.float32)
        lv = level3_valid > 0.5
        if lv.any():
            level3[lv] = (g_logs[lv] * first3_mask[lv]).sum(axis=1) / 3.0

        return {
            'z': z,
            'any30': any30,
            'od7': np.log1p(od7).astype(np.float32),
            'od14': np.log1p(od14).astype(np.float32),
            'od30': np.log1p(od30).astype(np.float32),
            'ad7': np.log1p(ad7).astype(np.float32),
            'ad30': np.log1p(ad30).astype(np.float32),
            'gap_cls': gap_class(first_gap),
            'repeat_gap_cls': gap_class(repeat_gap),
            'repeat_valid': any30.astype(np.float32),
            'first_mark': first_mark,
            'first_mark_valid': first_mark_valid,
            'level3': level3,
            'level3_valid': level3_valid,
            'order_days_raw': od30,
        }

    def torch_batch(self, users: np.ndarray, anchor: date):
        x = torch.from_numpy(self.history_np(users, anchor)).to(DEVICE, non_blocking=True)
        lab_np = self.labels_np(users, anchor)
        labs = {}
        for k,v in lab_np.items():
            t = torch.from_numpy(np.asarray(v))
            if k.endswith('_cls'):
                t = t.long()
            else:
                t = t.float()
            labs[k] = t.to(DEVICE, non_blocking=True)
        return x, labs

ACCESS = HistoryAccessor()
print('Accessor ready. INPUT_C=', INPUT_C, 'N_PATCHES=', N_PATCHES)


Scaler cutoff: 2025-06-18
Scaler ready
Accessor ready. INPUT_C= 24 N_PATCHES= 52


## Модель

In [5]:
class TCNBlock(nn.Module):
    def __init__(self, dim: int, dilation: int, dropout: float=0.10):
        super().__init__()
        self.conv1 = nn.Conv1d(dim, dim, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(dim, dim, 3, padding=dilation, dilation=dilation)
        self.norm1 = nn.GroupNorm(1, dim)
        self.norm2 = nn.GroupNorm(1, dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.conv1(x); x = self.norm1(x); x = F.silu(x); x = self.drop(x)
        x = self.conv2(x); x = self.norm2(x); x = F.silu(x); x = self.drop(x)
        return x + r

class RepresentationNet(nn.Module):
    def __init__(self, input_dim=INPUT_C, raw_dim=RAW_C, d_model=64, emb_dim=128, bridge_dim=BRIDGE_DIM):
        super().__init__()
        self.input_dim = input_dim
        self.raw_dim = raw_dim
        self.patch_proj = nn.Sequential(
            nn.Linear(PATCH_DAYS * input_dim, d_model), nn.LayerNorm(d_model), nn.GELU()
        )
        self.patch_pos = nn.Parameter(torch.zeros(1, N_PATCHES, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=192, dropout=0.10,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.long_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.day_proj = nn.Sequential(nn.Linear(input_dim, d_model), nn.LayerNorm(d_model), nn.SiLU())
        self.recent_tcn = nn.Sequential(*[TCNBlock(d_model, d) for d in [1,2,4,8,16]])
        fusion_in = d_model*5
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(256, emb_dim), nn.LayerNorm(emb_dim), nn.GELU(),
        )
        g = torch.Generator(device='cpu'); g.manual_seed(987654321)
        mat = torch.randn(emb_dim, bridge_dim, generator=g, dtype=torch.float32) / math.sqrt(float(emb_dim))
        self.register_buffer('bridge_matrix', mat)
        self.recon_head = nn.Linear(d_model, PATCH_DAYS * raw_dim)
        self.z_head = nn.Sequential(nn.Linear(emb_dim, 128), nn.GELU(), nn.Dropout(0.05), nn.Linear(128, 1))
        self.any_head = nn.Linear(emb_dim, 1)
        self.freq_head = nn.Linear(emb_dim, 3)
        self.active_head = nn.Linear(emb_dim, 2)
        self.gap_head = nn.Linear(emb_dim, 5)
        self.repeat_gap_head = nn.Linear(emb_dim, 5)
        self.first_mark_head = nn.Linear(emb_dim, 1)
        self.level3_head = nn.Linear(emb_dim, 1)
        nn.init.normal_(self.patch_pos, std=0.02)

    def encode(self, x):

        p = x[:, :LONG_DAYS, :].reshape(x.shape[0], N_PATCHES, PATCH_DAYS*self.input_dim)
        p = self.patch_proj(p) + self.patch_pos
        long_h = self.long_encoder(p)
        long_last = long_h[:, -1]
        long_mean = long_h.mean(dim=1)

        r = x[:, -RECENT_DAYS:, :]
        r = self.day_proj(r).transpose(1,2)
        r = self.recent_tcn(r).transpose(1,2)
        recent_last = r[:, -1]
        recent_mean = r.mean(dim=1)
        recent_max = r.amax(dim=1)
        emb = self.fusion(torch.cat([long_last,long_mean,recent_last,recent_mean,recent_max], dim=1))
        return emb, long_h

    def forward(self, x):
        emb, long_h = self.encode(x)
        return {
            'emb': emb,
            'bridge': emb @ self.bridge_matrix,
            'recon': self.recon_head(long_h),
            'z': self.z_head(emb).squeeze(-1),
            'any30': self.any_head(emb).squeeze(-1),
            'freq': self.freq_head(emb),
            'active': self.active_head(emb),
            'gap': self.gap_head(emb),
            'repeat_gap': self.repeat_gap_head(emb),
            'first_mark': self.first_mark_head(emb).squeeze(-1),
            'level3': self.level3_head(emb).squeeze(-1),
        }

    def reset_z_head(self):
        self.z_head = nn.Sequential(nn.Linear(self.fusion[-2].normalized_shape[0] if False else 128, 128), nn.GELU(), nn.Dropout(0.05), nn.Linear(128,1)).to(DEVICE)

def fresh_model() -> RepresentationNet:
    return RepresentationNet().to(DEVICE)

def reset_main_head(model: RepresentationNet):
    model.z_head = nn.Sequential(nn.Linear(128,128), nn.GELU(), nn.Dropout(0.05), nn.Linear(128,1)).to(DEVICE)

_rng_alignment_model = fresh_model()
del _rng_alignment_model
torch.cuda.empty_cache()


C:\Users\Никита\AppData\Local\Temp\ipykernel_17376\2120590018.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.long_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


In [6]:
def apply_active_weighted_patch_mask(x: torch.Tensor):
    target_raw = x[:, :LONG_DAYS, :RAW_C].reshape(x.shape[0], N_PATCHES, PATCH_DAYS*RAW_C).detach().clone()
    row = x[:, :LONG_DAYS, IDX_ROW_PRESENT].reshape(x.shape[0], N_PATCHES, PATCH_DAYS)
    active = (row.sum(dim=-1) > 0).float()
    valid_patch = (x[:, :LONG_DAYS, IDX_VALID].reshape(x.shape[0], N_PATCHES, PATCH_DAYS).sum(dim=-1) > 0).float()
    weights = (1.0 + 3.0*active) * valid_patch + 1e-4*(1.0-valid_patch)
    k = max(1, int(round(N_PATCHES * 0.20)))
    idx = torch.multinomial(weights, num_samples=k, replacement=False)
    patch_mask = torch.zeros((x.shape[0], N_PATCHES), device=x.device, dtype=torch.bool)
    patch_mask.scatter_(1, idx, True)
    day_mask = patch_mask.repeat_interleave(PATCH_DAYS, dim=1)
    xm = x.clone()
    first = xm[:, :LONG_DAYS, :]
    first[:, :, :RAW_C] = torch.where(day_mask[:,:,None], torch.zeros_like(first[:,:,:RAW_C]), first[:,:,:RAW_C])
    first[:, :, IDX_ROW_PRESENT] = torch.where(day_mask, torch.zeros_like(first[:,:,IDX_ROW_PRESENT]), first[:,:,IDX_ROW_PRESENT])
    first[:, :, IDX_MASKED] = torch.where(day_mask, torch.ones_like(first[:,:,IDX_MASKED]), first[:,:,IDX_MASKED])
    xm[:, :LONG_DAYS, :] = first
    return xm, target_raw, patch_mask

def masked_recon_loss(pred, target, patch_mask):
    diff = (pred - target).pow(2).mean(dim=-1)
    if patch_mask.any():
        return diff[patch_mask].mean()
    return diff.mean()*0.0

def supervised_aux_loss(out, lab, include_dense_z: bool):
    loss_any = F.binary_cross_entropy_with_logits(out['any30'], lab['any30'])
    freq_t = torch.stack([lab['od7'], lab['od14'], lab['od30']], dim=1)
    active_t = torch.stack([lab['ad7'], lab['ad30']], dim=1)
    loss_freq = F.mse_loss(out['freq'], freq_t)
    loss_active = F.mse_loss(out['active'], active_t)
    loss_gap = F.cross_entropy(out['gap'], lab['gap_cls'])
    rv = lab['repeat_valid'] > 0.5
    loss_repeat = F.cross_entropy(out['repeat_gap'][rv], lab['repeat_gap_cls'][rv]) if rv.any() else out['repeat_gap'].sum()*0.0
    fm = lab['first_mark_valid'] > 0.5
    loss_mark = F.mse_loss(out['first_mark'][fm], lab['first_mark'][fm]) if fm.any() else out['first_mark'].sum()*0.0
    lv = lab['level3_valid'] > 0.5
    loss_level = F.mse_loss(out['level3'][lv], lab['level3'][lv]) if lv.any() else out['level3'].sum()*0.0
    total = (
        W_ANY30*loss_any + W_FREQ*loss_freq + W_ACTIVE*loss_active + W_GAP*loss_gap
        + W_REPEAT_GAP*loss_repeat + W_FIRST_MARK*loss_mark + W_LEVEL3*loss_level
    )
    loss_z = F.mse_loss(out['z'], lab['z']) if include_dense_z else out['z'].sum()*0.0
    if include_dense_z:
        total = total + W_DENSE_Z*loss_z
    return total, {
        'any': float(loss_any.detach()), 'freq': float(loss_freq.detach()), 'active': float(loss_active.detach()),
        'gap': float(loss_gap.detach()), 'repeat_gap': float(loss_repeat.detach()), 'mark': float(loss_mark.detach()),
        'level3': float(loss_level.detach()), 'dense_z': float(loss_z.detach()),
    }

def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)


In [7]:
def available_origin_indices(cutoff: date, min_history_days: int=0) -> np.ndarray:
    hi = date_idx(cutoff) - HORIZON
    lo = max(0, min_history_days)
    if hi < lo:
        return np.array([], dtype=np.int64)
    return np.arange(lo, hi+1, dtype=np.int64)

def origin_date_from_idx(i: int) -> date:
    return min_date + timedelta(days=int(i))

def train_pretrain_stage(model: RepresentationNet, cutoff: date, steps: int, include_dense_z: bool, stage_name: str):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    origins = available_origin_indices(cutoff)
    if len(origins) == 0:
        print(stage_name, cutoff, 'skip')
        return
    rng = np.random.default_rng(SEED + date_idx(cutoff) + (1000 if include_dense_z else 0))
    t0 = time.time()
    for step in range(steps):
        oi = int(rng.choice(origins)); a = origin_date_from_idx(oi)
        users = rng.integers(0, EXPECTED_USERS, size=BUDGET['batch'], dtype=np.int64)
        x, lab = ACCESS.torch_batch(users, a)
        xm, target_raw, pmask = apply_active_weighted_patch_mask(x)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            out = model(xm)
            lrec = masked_recon_loss(out['recon'], target_raw, pmask)
            laux, parts = supervised_aux_loss(out, lab, include_dense_z=include_dense_z)
            loss = W_RECON*lrec + laux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(opt); scaler.update()
        if step % max(1, steps//8) == 0 or step == steps-1:
            row = {'stage':stage_name,'cutoff':str(cutoff),'step':step,'loss':float(loss.detach()),'recon':float(lrec.detach()),**parts}
            PRETRAIN_LOG_ROWS.append(row)
            print(row)
        del x, lab, xm, target_raw, pmask, out, loss
    print(stage_name, cutoff, 'seconds=', round(time.time()-t0,1))
    del opt, scaler
    gc.collect(); torch.cuda.empty_cache()


In [8]:
def predict_bridge_subset(model: RepresentationNet, anchor: date, users: np.ndarray):
    model.eval(); out_parts=[]
    users=np.asarray(users,dtype=np.int64)
    with torch.no_grad():
        for s in range(0,len(users),BUDGET['infer_chunk']):
            uu=users[s:s+BUDGET['infer_chunk']]
            x=torch.from_numpy(ACCESS.history_np(uu,anchor)).to(DEVICE,non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                out=model(x)
            out_parts.append(out['bridge'].float().cpu().numpy())
            del x,out
    return np.concatenate(out_parts).astype(np.float32)

def predict_bridge_all(model: RepresentationNet, anchor: date) -> np.ndarray:
    model.eval()
    parts = []

    with torch.no_grad():
        for s in range(0, EXPECTED_USERS, BUDGET["infer_chunk"]):
            e = min(EXPECTED_USERS, s + BUDGET["infer_chunk"])
            users = np.arange(s, e, dtype=np.int64)
            x = torch.from_numpy(
                ACCESS.history_np(users, anchor)
            ).to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=USE_AMP,
            ):
                out = model(x)

            parts.append(
                out["bridge"].float().cpu().numpy()
            )

            del x, out

    return np.concatenate(parts).astype(np.float32)


## Обучение и экспорт

In [9]:
PRETRAIN_LOG_ROWS = []
REP_CUTOFFS = sorted(set(V9_CANONICAL_POOL + [max_date]))

behavior_model = fresh_model()
dense_model = fresh_model()

ALIGN_USERS = np.random.default_rng(SEED + 991).choice(
    EXPECTED_USERS,
    size=20_000,
    replace=False,
).astype(np.int64)

BRIDGE_ROT = np.eye(BRIDGE_DIM, dtype=np.float32)
PREV_STAGE_CUTOFF = None
CURRENT_ALIGNMENT_RMSE = 0.0
BRIDGE_ROWS = []

np.save(
    BRIDGE_DIR / "user_ids.npy",
    all_users["user_id"].cast(pl.Int64).to_numpy(),
)

def save_quantized_bridge(
    anchor: date,
    emb: np.ndarray,
) -> None:
    scale = (
        np.max(np.abs(emb), axis=0)
        .astype(np.float32)
        / 127.0
    )
    scale = np.where(
        scale < 1e-8,
        1.0,
        scale,
    ).astype(np.float32)

    q = np.clip(
        np.rint(emb / scale[None, :]),
        -127,
        127,
    ).astype(np.int8)

    path = BRIDGE_DIR / f"repr_dense_{anchor}.npz"
    np.savez_compressed(
        path,
        q=q,
        scale=scale,
    )

    recon = q.astype(np.float32) * scale[None, :]
    quant_rmse = float(
        np.sqrt(
            np.mean(
                (emb - recon) ** 2
            )
        )
    )

    BRIDGE_ROWS.append({
        "anchor": str(anchor),
        "file": path.name,
        "rows": int(emb.shape[0]),
        "dims": int(emb.shape[1]),
        "quant_rmse": quant_rmse,
        "alignment_rmse": float(
            CURRENT_ALIGNMENT_RMSE
        ),
    })

    print(
        anchor,
        path.name,
        f"quant_rmse={quant_rmse:.6f}",
    )

for stage_i, cutoff in enumerate(REP_CUTOFFS, 1):
    print(
        f"[{stage_i}/{len(REP_CUTOFFS)}] {cutoff}"
    )

    prev_dense = (
        copy.deepcopy(dense_model).eval()
        if PREV_STAGE_CUTOFF is not None
        else None
    )
    prev_rot = BRIDGE_ROT.copy()

    train_pretrain_stage(
        behavior_model,
        cutoff,
        BUDGET["pretrain_steps"],
        include_dense_z=False,
        stage_name="BEHAV_MASK_AUX",
    )

    train_pretrain_stage(
        dense_model,
        cutoff,
        BUDGET["dense_steps"],
        include_dense_z=True,
        stage_name="DENSE_GMV_MASK_AUX",
    )

    if prev_dense is not None:
        old_raw = predict_bridge_subset(
            prev_dense,
            PREV_STAGE_CUTOFF,
            ALIGN_USERS,
        )
        new_raw = predict_bridge_subset(
            dense_model,
            PREV_STAGE_CUTOFF,
            ALIGN_USERS,
        )
        old_global = old_raw @ prev_rot

        m = new_raw.T @ old_global
        u, _, vt = np.linalg.svd(
            m,
            full_matrices=False,
        )
        BRIDGE_ROT = (
            u @ vt
        ).astype(np.float32)

        CURRENT_ALIGNMENT_RMSE = float(
            np.sqrt(
                np.mean(
                    (
                        new_raw @ BRIDGE_ROT
                        - old_global
                    ) ** 2
                )
            )
        )

        del (
            old_raw,
            new_raw,
            old_global,
            m,
            u,
            vt,
            prev_dense,
        )
        gc.collect()
        torch.cuda.empty_cache()

    bridge = predict_bridge_all(
        dense_model,
        cutoff,
    )
    bridge = bridge @ BRIDGE_ROT

    save_quantized_bridge(
        cutoff,
        bridge,
    )

    del bridge
    gc.collect()
    torch.cuda.empty_cache()

    PREV_STAGE_CUTOFF = cutoff

manifest = {
    "version": "neural_features_v24r_dense_pretrain",
    "representation": "DENSE_GMV_MASK_AUX",
    "bridge_dim": BRIDGE_DIM,
    "raw_channels": RAW_COLS,
    "lookback_days": LOOKBACK,
    "patch_days": PATCH_DAYS,
    "recent_days": RECENT_DAYS,
    "anchors": [
        str(a)
        for a in REP_CUTOFFS
    ],
    "user_ids_file": "user_ids.npy",
    "files": BRIDGE_ROWS,
    "quantization": "per-dimension symmetric int8",
    "coordinate_alignment": "sequential orthogonal Procrustes",
    "seed": SEED,
}

(BRIDGE_DIR / "manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

if PRETRAIN_LOG_ROWS:
    pl.DataFrame(
        PRETRAIN_LOG_ROWS
    ).write_csv(
        ARTIFACT_ROOT / "training_log.csv"
    )

expected_files = {
    f"repr_dense_{anchor}.npz"
    for anchor in REP_CUTOFFS
}
actual_files = {
    p.name
    for p in BRIDGE_DIR.glob(
        "repr_dense_*.npz"
    )
}

assert expected_files <= actual_files
assert (
    BRIDGE_DIR / "user_ids.npy"
).is_file()
assert (
    BRIDGE_DIR / "manifest.json"
).is_file()
assert len(BRIDGE_ROWS) == len(REP_CUTOFFS)
assert all(
    row["rows"] == EXPECTED_USERS
    for row in BRIDGE_ROWS
)
assert all(
    row["dims"] == BRIDGE_DIM
    for row in BRIDGE_ROWS
)

print("Representation:", BRIDGE_DIR.resolve())
print("Anchors:", len(REP_CUTOFFS))


C:\Users\Никита\AppData\Local\Temp\ipykernel_17376\2120590018.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.long_encoder = nn.TransformerEncoder(enc_layer, num_layers=2)


[1/12] 2025-02-13
{'stage': 'BEHAV_MASK_AUX', 'cutoff': '2025-02-13', 'step': 0, 'loss': 3.415154218673706, 'recon': 0.4098207950592041, 'any': 0.6695500612258911, 'freq': 0.5457448959350586, 'active': 3.559022903442383, 'gap': 1.9173107147216797, 'repeat_gap': 1.6255090236663818, 'mark': 15.322476387023926, 'level3': 12.899391174316406, 'dense_z': -0.0}
{'stage': 'BEHAV_MASK_AUX', 'cutoff': '2025-02-13', 'step': 50, 'loss': 0.7387977838516235, 'recon': 0.14503872394561768, 'any': 0.6549721956253052, 'freq': 0.24642162024974823, 'active': 0.8081573843955994, 'gap': 1.1787910461425781, 'repeat_gap': 1.5278269052505493, 'mark': 1.2403225898742676, 'level3': 0.4774768054485321, 'dense_z': -0.0}
{'stage': 'BEHAV_MASK_AUX', 'cutoff': '2025-02-13', 'step': 100, 'loss': 0.6838228106498718, 'recon': 0.1427748054265976, 'any': 0.6118469834327698, 'freq': 0.20282301306724548, 'active': 0.7685319185256958, 'gap': 1.0923891067504883, 'repeat_gap': 1.4188356399536133, 'mark': 1.0419154167175293, 'l